# Plot Checkpoint Predictions - v22

Select a trained v22 checkpoint, one session date, and a ticker list. The notebook rebuilds chronological event-chunk windows, runs inference, and plots predicted midpoint vs target midpoint.

In [ ]:
from pathlib import Path
import os

MODEL_VERSION = "v22"


def find_workspace(model_version: str) -> Path:
    """Find the repo/runtime root on either the laptop checkout or workstation runtime."""
    candidates: list[Path] = []
    for env_name in ("QUANT_RESEARCH_WORKSPACE", "QRW_WORKSPACE"):
        value = os.environ.get(env_name)
        if value:
            candidates.append(Path(value))

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend(
        [
            Path(r"D:/TradingCodes/quant-research-workbench-v22-runtime"),
            Path(r"D:/TradingCodes/quant-research-workbench"),
            Path(r"//DESKTOP-SAAI85T/Workstation-D/TradingCodes/quant-research-workbench-v22-runtime"),
        ]
    )

    checked: list[str] = []
    for candidate in candidates:
        root = candidate.resolve() if candidate.exists() else candidate
        marker = root / "research" / "inhouse_transformer" / model_version / "config.py"
        checked.append(str(marker))
        if marker.exists():
            return root
    raise FileNotFoundError(
        "Could not find a repo root containing "
        f"research/inhouse_transformer/{model_version}/config.py. "
        "Set QUANT_RESEARCH_WORKSPACE to the checkout/runtime root. Checked: "
        + "; ".join(checked[:12])
    )


WORKSPACE = find_workspace(MODEL_VERSION)
NOTEBOOK_DIR = WORKSPACE / "research" / "inhouse_transformer" / MODEL_VERSION
MODEL_ROOT = Path(r"D:/TradingData/quant-research-workbench/market_data/models/inhouse_transformer") / MODEL_VERSION
print(f"WORKSPACE={WORKSPACE}")
print(f"NOTEBOOK_DIR={NOTEBOOK_DIR}")

# Edit these values for the batch/checkpoint you want to inspect.
CHECKPOINT_PATH = ""  # leave empty to use newest last.pt under MODEL_ROOT

SESSION_DATE = "2025-11-03"
TICKERS = ("A", "AA", "AAA")
PLOT_HORIZON_INDEX = 0
MAX_WINDOWS_PER_TICKER = 600
BATCH_SIZE = 128
DEVICE = "cuda"

FLATFILES_ROOT = Path(r"D:/market-data/flatfiles/us_stocks_sip")
CANONICAL_ROOT = Path(r"D:/market-data/flatfiles/us_stocks_sip/derived/canonical_events_v1")
CACHE_ROOT = Path(r"D:/market-data/flatfiles/us_stocks_sip/derived/event_chunks_v1")


In [ ]:

import sys
from dataclasses import fields
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from research.inhouse_transformer.v22.config import DataConfig, ModelConfig
from research.inhouse_transformer.v22.data import (
    CHUNK_SUMMARY_COLUMNS,
    QUOTE_FEATURE_COLUMNS,
    TRADE_FEATURE_COLUMNS,
    EventChunkDataset,
    available_sessions,
)
from research.inhouse_transformer.v22.model import HierarchicalEventTransformer
from research.inhouse_transformer.v22.targets import target_values_to_bps

PATH_FIELDS = ("flatfiles_root", "canonical_root", "cache_root")
TUPLE_FIELDS = ("tickers", "target_columns", "target_cache_horizon_chunks")

def dataclass_from_dict(cls, payload):
    allowed = {field.name for field in fields(cls)}
    values = {key: value for key, value in dict(payload or {}).items() if key in allowed}
    for key in PATH_FIELDS:
        if key in values:
            values[key] = Path(values[key])
    for key in TUPLE_FIELDS:
        if key in values and isinstance(values[key], list):
            values[key] = tuple(values[key])
    return cls(**values)


def resolve_checkpoint(path_text, model_root):
    if path_text:
        path = Path(path_text)
        if not path.exists():
            raise FileNotFoundError(path)
        return path
    candidates = sorted(model_root.glob("**/last.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f"No last.pt found under {model_root}. Set CHECKPOINT_PATH explicitly.")
    return candidates[0]


def direction_accuracy(pred_bps, target_bps, threshold=0.0):
    pred_dir = np.sign(np.where(np.abs(pred_bps) <= threshold, 0.0, pred_bps))
    target_dir = np.sign(np.where(np.abs(target_bps) <= threshold, 0.0, target_bps))
    return float((pred_dir == target_dir).mean() * 100.0)


In [ ]:

checkpoint_path = resolve_checkpoint(CHECKPOINT_PATH, MODEL_ROOT)
checkpoint = torch.load(checkpoint_path, map_location="cpu")
data_config = dataclass_from_dict(DataConfig, checkpoint.get("config", {}).get("data", {}))
model_config = dataclass_from_dict(ModelConfig, checkpoint.get("config", {}).get("model", {}))
data_config.flatfiles_root = FLATFILES_ROOT
data_config.canonical_root = CANONICAL_ROOT
data_config.cache_root = CACHE_ROOT
data_config.tickers = TICKERS
data_config.max_windows_per_ticker_session = MAX_WINDOWS_PER_TICKER
model_config.target_bit_count = 1 + int(data_config.binary_magnitude_bits)

model = HierarchicalEventTransformer(
    quote_feature_count=len(QUOTE_FEATURE_COLUMNS),
    trade_feature_count=len(TRADE_FEATURE_COLUMNS),
    chunk_summary_count=len(CHUNK_SUMMARY_COLUMNS),
    context_chunks=data_config.context_chunks,
    max_quote_events=data_config.max_quote_events,
    max_trade_events=data_config.max_trade_events,
    max_total_events=data_config.max_total_events,
    horizon_steps=data_config.target_horizon_count,
    target_count=len(data_config.target_columns),
    config=model_config,
)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
device = torch.device(DEVICE if DEVICE == "cuda" and torch.cuda.is_available() else "cpu")
model = model.to(device).eval()
print("Loaded", checkpoint_path, "step", checkpoint.get("global_step"))
print("target horizons seconds:", data_config.target_horizon_seconds)


In [ ]:

# Keras-style model summary and graph view.
# Optional packages for richer output:
#   pip install torchinfo torchview graphviz
# Graph rendering also needs the Graphviz dot executable on PATH.

from IPython.display import Image, Markdown, display
import shutil

param_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"parameters={param_count:,} trainable={trainable_count:,}")

try:
    from torchinfo import summary
    print(summary(model, input_data=(torch.zeros(1, data_config.context_chunks, data_config.max_quote_events, len(QUOTE_FEATURE_COLUMNS), device=device), torch.zeros(1, data_config.context_chunks, data_config.max_trade_events, len(TRADE_FEATURE_COLUMNS), device=device), torch.full((1, data_config.context_chunks, data_config.max_total_events), 2, dtype=torch.long, device=device), torch.zeros(1, data_config.context_chunks, data_config.max_total_events, dtype=torch.long, device=device), torch.zeros(1, data_config.context_chunks, len(CHUNK_SUMMARY_COLUMNS), device=device)), depth=8, col_names=("input_size", "output_size", "num_params"), verbose=0))
except Exception as exc:
    print("torchinfo summary unavailable:", repr(exc))
    print(model)

try:
    from torchview import draw_graph
    dot = shutil.which("dot")
    if dot is None:
        raise RuntimeError("Graphviz dot executable is not on PATH. Install Graphviz and add its bin folder to PATH.")
    graph = draw_graph(model, input_data=(torch.zeros(1, data_config.context_chunks, data_config.max_quote_events, len(QUOTE_FEATURE_COLUMNS), device=device), torch.zeros(1, data_config.context_chunks, data_config.max_trade_events, len(TRADE_FEATURE_COLUMNS), device=device), torch.full((1, data_config.context_chunks, data_config.max_total_events), 2, dtype=torch.long, device=device), torch.zeros(1, data_config.context_chunks, data_config.max_total_events, dtype=torch.long, device=device), torch.zeros(1, data_config.context_chunks, len(CHUNK_SUMMARY_COLUMNS), device=device)), expand_nested=True, save_graph=True, filename=f"{MODEL_VERSION}_architecture", directory=str(NOTEBOOK_DIR))
    png_path = NOTEBOOK_DIR / f"{MODEL_VERSION}_architecture.png"
    display(Image(filename=str(png_path)))
except Exception as exc:
    display(Markdown(f"Graph rendering skipped: `{exc!r}`"))


In [ ]:

sessions = available_sessions(data_config.flatfiles_root, SESSION_DATE, SESSION_DATE)
dataset = EventChunkDataset(
    config=data_config,
    sessions=sessions,
    tickers=TICKERS,
    batch_size=BATCH_SIZE,
    seed=17,
    mode="plot",
    epochs=1,
    max_windows=max(1, len(TICKERS)) * MAX_WINDOWS_PER_TICKER,
    shuffle=False,
)

rows = []
with torch.no_grad():
    for batch in dataset:
        prediction = model(
            batch["quote_values"].to(device),
            batch["trade_values"].to(device),
            batch["event_kinds"].to(device),
            batch["event_indices"].to(device),
            batch["chunk_summary"].to(device),
        )
        current_mid = batch["current_mid"].numpy()
        pred_bps = target_values_to_bps(
            prediction.detach().cpu().numpy(),
            current_mid,
            np.zeros_like(current_mid),
            np.ones_like(current_mid),
            data_config.target_mode,
        )[..., 0]
        target_bps = batch["target_bps"].numpy()[..., 0]
        target_mid = batch["target_mid"].numpy()[..., 0]
        origin = pd.to_datetime(batch["origin_timestamp_ns"].numpy(), unit="ns", utc=True)
        for row_idx, ticker in enumerate(batch["ticker"]):
            for horizon_index, horizon_seconds in enumerate(data_config.target_horizon_seconds):
                rows.append({
                    "ticker": ticker,
                    "origin_time_utc": origin[row_idx],
                    "horizon_index": horizon_index,
                    "horizon_seconds": float(horizon_seconds),
                    "current_mid": float(current_mid[row_idx]),
                    "target_bps": float(target_bps[row_idx, horizon_index]),
                    "pred_bps": float(pred_bps[row_idx, horizon_index]),
                    "target_mid": float(target_mid[row_idx, horizon_index]),
                    "pred_mid": float(current_mid[row_idx] * np.exp(pred_bps[row_idx, horizon_index] / 10000.0)),
                })

predictions = pd.DataFrame(rows).sort_values(["ticker", "origin_time_utc", "horizon_index"])
predictions.head()


In [ ]:

plot_df = predictions[predictions["horizon_index"] == PLOT_HORIZON_INDEX].copy()
metrics = []
for (ticker, horizon_seconds), group in predictions.groupby(["ticker", "horizon_seconds"]):
    metrics.append({
        "ticker": ticker,
        "horizon_seconds": horizon_seconds,
        "windows": len(group),
        "mae_bps": float(np.mean(np.abs(group["pred_bps"] - group["target_bps"]))),
        "rmse_bps": float(np.sqrt(np.mean(np.square(group["pred_bps"] - group["target_bps"])))),
        "dir_acc_pct": direction_accuracy(group["pred_bps"].to_numpy(), group["target_bps"].to_numpy()),
    })
metrics_df = pd.DataFrame(metrics).sort_values(["ticker", "horizon_seconds"])
display(metrics_df)

for ticker, group in plot_df.groupby("ticker"):
    fig, ax = plt.subplots(figsize=(16, 5))
    group = group.sort_values("origin_time_utc")
    ax.plot(group["origin_time_utc"], group["target_mid"], label="target_mid", linewidth=1.8)
    ax.plot(group["origin_time_utc"], group["pred_mid"], label="pred_mid", linewidth=1.0, linestyle="--")
    ax.set_title(f"{MODEL_VERSION} {ticker} horizon={group['horizon_seconds'].iloc[0]:g}s")
    ax.set_xlabel("origin time UTC")
    ax.set_ylabel("mid price")
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.show()
